# Dá pra morar em Lucas? Preço dos imóveis x renda — a conta da acessibilidade

**Autora:** Thamires Martins da Silva  
**Fontes:** Dataset de preços da Camada 3 (**233 anúncios após remoção de duplicatas exatas**) · Salário mínimo 2026 · Rendimento médio de Mato Grosso (PNAD Contínua/IBGE, 2024)  
**Ferramentas:** Python (pandas, matplotlib)

---

## Contexto

Este é o quarto projeto de uma série conectada sobre habitação em Lucas do Rio Verde. As camadas anteriores estimaram a demanda habitacional futura, descreveram a oferta de interesse social e mapearam preços anunciados. Esta camada compara a amostra de preços com **faixas de renda de referência** para calcular indicadores ilustrativos de acessibilidade.

O IBGE não publica, na base utilizada neste projeto, uma renda familiar municipal fechada para Lucas. Por isso, a análise não estima quantas famílias reais da cidade conseguem comprar um imóvel. Em vez disso, avalia como a acessibilidade muda em cenários de renda familiar.

## Pergunta de negócio

**Para diferentes faixas de renda familiar de referência, que parcela dos imóveis anunciados poderia caber no orçamento de uma simulação de financiamento?**

## Nota de método sobre a renda

Os cenários usam o salário mínimo nacional de 2026 (**R\$ 1.621**) e o rendimento médio mensal individual de Mato Grosso informado para a PNAD Contínua/IBGE de 2024 (**R\$ 3.402 por pessoa**) como referências. Essas grandezas não são equivalentes a uma renda familiar municipal observada; as faixas salariais são hipóteses para testar cenários, não uma distribuição medida de renda local.

Os cálculos de financiamento assumem entrada, prazo, taxa e limite de comprometimento de renda definidos no notebook. Eles não representam aprovação bancária, proposta comercial ou simulação personalizada.

## 1. Configuração e carga

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

AZUL_ESC='#1a3a5c'; AZUL='#2c7fb8'; TEAL='#7fcdbb'; LARANJA='#e8703a'; CINZA='#8a9299'

def reais(v): return ('R$ '+format(int(round(v)),',d')).replace(',','.')

SALARIO_MINIMO = 1621          # 2026
REND_MEDIO_MT  = 3402          # PNAD Continua/IBGE 2024, por pessoa

df = pd.read_csv(Path.cwd()/'precos_imoveis_lucas.csv', sep=';')
duplicatas_exatas = int(df.duplicated().sum())
if duplicatas_exatas:
    df = df.drop_duplicates().copy()

# Combinações iguais entre fontes diferentes são candidatas a revisão,
# mas não são removidas automaticamente sem identificação do imóvel.
chave_candidata = ['bairro', 'tipo', 'area_m2', 'preco_total', 'preco_m2']
candidatos_duplicacao = df[df.duplicated(subset=chave_candidata, keep=False)]
print(f'Duplicatas exatas removidas ao carregar: {duplicatas_exatas}')
print(f'Registros com possível sobreposição entre fontes: {len(candidatos_duplicacao)}')

tip = df[df['atipico']=='nao'].copy()
preco_tipico = tip['preco_total'].median()
print(f'Anuncios tipicos: {len(tip)}')
print(f'Imovel tipico (mediana):    {reais(preco_tipico)}')
print(f'  casa:  {reais(tip[tip.tipo=="casa"]["preco_total"].median())}')
print(f'  apto:  {reais(tip[tip.tipo=="apartamento"]["preco_total"].median())}')
print(f'Salario minimo 2026:        {reais(SALARIO_MINIMO)}')
print(f'Rendimento medio MT/pessoa: {reais(REND_MEDIO_MT)}')

## 2. Quantos anos de renda custa um imóvel

In [ ]:
def anos_de_renda(preco, renda_mensal):
    return preco/(renda_mensal*12)

print('=== ANOS DE RENDA FAMILIAR INTEGRAL PARA O IMOVEL TIPICO ===')
print('   (renda inteira, sem descontar nenhum outro gasto)\n')
faixas_sm=[2,3,4,6,8,10]
for sm in faixas_sm:
    renda=sm*SALARIO_MINIMO
    print(f'  {sm:2} sal.min.  ({reais(renda)}/mes)  ->  {anos_de_renda(preco_tipico,renda):4.1f} anos')
print()
print(f'Referencia: ao rendimento medio individual de MT ({reais(REND_MEDIO_MT)}),')
print(f'o imovel tipico equivale a {anos_de_renda(preco_tipico,REND_MEDIO_MT):.0f} anos de renda.')

In [ ]:
rendas=np.array([2,3,4,5,6,8,10])*SALARIO_MINIMO
anos=[anos_de_renda(preco_tipico,r) for r in rendas]
fig,ax=plt.subplots(figsize=(11,5.5))
cores=[LARANJA if a>5 else TEAL for a in anos]
ax.bar([f'{int(r/SALARIO_MINIMO)} SM' for r in rendas],anos,color=cores,zorder=3)
for i,a in enumerate(anos):
    ax.annotate(f'{a:.1f} anos',(i,a),textcoords='offset points',xytext=(0,6),ha='center',
                fontsize=10,fontweight='bold',color=AZUL_ESC)
ax.axhline(5,color=CINZA,linestyle='--',linewidth=1.5,zorder=2)
ax.annotate('Linha de 5 anos de renda:\nacima disso, mercado severamente inacessivel\n(referencia internacional Demographia)',
            (len(rendas)-1,5),textcoords='offset points',xytext=(-6,12),ha='right',fontsize=8.5,color=CINZA)
ax.set_title('Quantos anos de renda familiar integral custa o imovel tipico de Lucas',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Renda familiar (em salarios minimos)'); ax.set_ylabel('Anos de renda integral')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
plt.tight_layout(); plt.show()

### Leitura

A referência internacional de cinco anos de renda é uma métrica agregada e não substitui uma análise local de crédito. Nos cenários modelados, o imóvel mediano equivale a cerca de **6,9 anos de renda integral** para uma família de quatro salários mínimos e **3,5 anos** para uma família de oito salários mínimos. Para duas faixas de renda mínima, o resultado não mede a proporção de famílias reais da cidade, pois não temos a distribuição de renda familiar municipal na base.

## 3. A conta real: financiar o imóvel típico

In [ ]:
def parcela_price(valor_financiado, taxa_aa=0.11, meses=360):
    i=(1+taxa_aa)**(1/12)-1
    return valor_financiado*i/(1-(1+i)**-meses)

ENTRADA=0.20
financiado=preco_tipico*(1-ENTRADA)
print(f'Imovel tipico: {reais(preco_tipico)}')
print(f'Entrada (20%): {reais(preco_tipico*ENTRADA)}   |   Financiado: {reais(financiado)}')
print(f'Prazo: 360 meses (30 anos)   |   Sistema: Price\n')
print('=== PARCELA E RENDA NECESSARIA (por taxa de juros) ===')
for taxa in [0.10,0.11,0.12]:
    p=parcela_price(financiado,taxa)
    renda_nec=p/0.30   # parcela <= 30% da renda
    print(f'  taxa {int(taxa*100)}% a.a.  ->  parcela {reais(p)}/mes  ->  renda familiar necessaria {reais(renda_nec)}  ({renda_nec/SALARIO_MINIMO:.1f} sal.min.)')

In [ ]:
taxas=[0.10,0.11,0.12]
parcelas=[parcela_price(financiado,t) for t in taxas]
rendas_nec=[p/0.30 for p in parcelas]
fig,ax=plt.subplots(figsize=(11,5.5))
x=np.arange(len(taxas))
ax.bar(x,rendas_nec,color=AZUL,width=.55,zorder=3)
for i,(rn,p) in enumerate(zip(rendas_nec,parcelas)):
    ax.annotate(f'{reais(rn)}\n(parcela {reais(p)})',(i,rn),textcoords='offset points',xytext=(0,6),
                ha='center',fontsize=10,fontweight='bold',color=AZUL_ESC)
ax.set_xticks(x); ax.set_xticklabels([f'{int(t*100)}% a.a.' for t in taxas])
ax.set_title('Renda familiar necessaria para financiar o imovel tipico (R$ 540 mil)',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Taxa de juros do financiamento'); ax.set_ylabel('Renda familiar necessaria (R$/mes)')
ax.set_ylim(0,max(rendas_nec)*1.25)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,p:f'{v/1000:.0f}k'))
plt.tight_layout(); plt.show()

### Leitura

Sob as premissas adotadas — entrada de 20%, prazo de 30 anos, sistema Price e juros de 10% a 12% a.a. — financiar o imóvel mediano exige uma renda familiar simulada de aproximadamente **R\$ 12,2 mil a R\$ 14,1 mil por mês**, além da entrada de R\$ 108 mil. A faixa não é uma condição real de crédito: seguros, tarifas, perfil do comprador, programa habitacional e critérios bancários podem alterar o resultado.

## 4. E o que cabe em cada renda?

In [ ]:
def maior_imovel(renda_mensal, ENTRADA=0.20, taxa=0.11):
    # maior valor de imovel cuja parcela cabe em 30% da renda, dada a entrada
    parcela_max=0.30*renda_mensal
    i=(1+taxa)**(1/12)-1
    financiado_max=parcela_max*(1-(1+i)**-360)/i
    return financiado_max/(1-ENTRADA)

print('=== TETO DE IMOVEL FINANCIAVEL POR RENDA (entrada 20%, 11% a.a., 30 anos) ===\n')
for sm in [2,3,4,6,8,10]:
    renda=sm*SALARIO_MINIMO
    teto=maior_imovel(renda)
    pct=(tip['preco_total']<=teto).mean()*100
    print(f'  {sm:2} SM ({reais(renda)}/mes)  ->  ate {reais(teto):>14}   | alcanca {pct:4.0f}% dos imoveis da amostra')

### Leitura

A simulação evidencia como o acesso à amostra muda conforme a renda de referência. Com quatro salários mínimos, o teto de preço calculado alcança cerca de **2% dos anúncios típicos**; com oito, aproximadamente **50%**; com dez, cerca de **74%**. Esses percentuais se referem à amostra de anúncios e às premissas do modelo — não indicam quantas famílias de Lucas do Rio Verde teriam aprovação para financiar esses imóveis.

## 5. Limitações

1. **Preço de anúncio, não de venda.** O valor pedido pode diferir do fechamento.
2. **Sem renda familiar municipal fechada.** A análise usa faixas hipotéticas, salário mínimo e rendimento médio individual de Mato Grosso; não estima a distribuição de renda das famílias de Lucas do Rio Verde.
3. **Parâmetros de financiamento assumidos.** Entrada de 20%, prazo de 30 anos, juros de 10–12% a.a., sistema Price e parcela limitada a 30% da renda. Condições reais variam por banco, programa e perfil.
4. **Renda familiar, não individual.** Os cenários somam a renda hipotética da família; o número de pessoas que contribuem por domicílio não é observado.
5. **Amostra de anúncios.** A base pode conter anúncios do mesmo imóvel em fontes diferentes; três linhas integralmente duplicadas foram removidas e combinações suspeitas foram mantidas para revisão manual.

## 6. Síntese executiva

Na amostra de **233 anúncios** da Camada 3, o preço mediano do imóvel típico é de **R\$ 540 mil**. Sob as premissas do modelo, uma simulação com entrada de 20%, prazo de 30 anos e juros de 11% a.a. resulta em renda familiar necessária de aproximadamente **R\$ 13,2 mil por mês** e entrada de **R\$ 108 mil**.

Medido como anos de renda integral, o imóvel típico equivale a cerca de **13,9 anos** para uma família hipotética de dois salários mínimos, **6,9 anos** para quatro salários mínimos e **3,5 anos** para oito salários mínimos. Na simulação de preço financiável, quatro salários mínimos alcançam aproximadamente 2% da amostra, oito salários mínimos 50% e dez salários mínimos 74%.

Esses resultados descrevem cenários e a amostra de anúncios, não a renda real das famílias do município ou uma aprovação bancária. Uma conclusão precisa sobre a acessibilidade de toda a população exigiria uma distribuição atual de renda familiar local e condições reais de financiamento.

---

*Quarta camada da análise sobre habitação em Lucas do Rio Verde. Baseada em 233 anúncios de 13 fontes e em referências de renda explicitadas na metodologia.*